# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Retrieve and show metadata
ds_metadata_dict = dataset.metadata.to_json()
ds_title = ds_metadata_dict.get('name', 'Unnamed Dataset')
ds_description = ds_metadata_dict.get('description', '')
print(f"{ds_title}: {ds_description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List record sets and their fields by @id
record_sets = []
metadata_dict = dataset.metadata.to_json()

# Check for Croissant 1.1+ convention for recordSet or fallback to fileObject if none.
if 'recordSet' in metadata_dict and len(metadata_dict['recordSet']) > 0:
    # The `recordSet` property contains a list of record sets, which might be dicts with '@id' keys
    for rec in metadata_dict['recordSet']:
        rs_id = rec['@id'] if isinstance(rec, dict) and '@id' in rec else rec
        record_sets.append(rs_id)
else:
    # As per the FAIR^2 schema, try to guess from 'distribution'->'@id' (files with tabular data)
    # and see if fileObject/recordSets must be inferred.
    if 'distribution' in metadata_dict:
        # We'll assume distribution refers to one or more files which are tabular, and load accordingly
        for dist in metadata_dict['distribution']:
            dist_id = dist['@id'] if isinstance(dist, dict) and '@id' in dist else dist
            record_sets.append(dist_id)

print("Record sets by @id:")
for rs in record_sets:
    print('-', rs)

if len(record_sets) == 0:
    print("No explicit record sets found. This dataset might use default file-based access.\n")
    print("You can try to directly use `dataset.records()` to enumerate available data instances.")
else:
    # List fields in the first record set
    print(f"\nExample fields for the first record set ({record_sets[0]}):")
    try:
        for idx, rec in enumerate(dataset.records(record_set=record_sets[0])):
            if idx > 2:
                break
            print(f"Record example {idx+1}:")
            print(rec)
    except Exception as e:
        print(f"Could not enumerate records for record set {record_sets[0]}: {e}")
        print("Try using dataset.records() without arguments or consult the dataset documentation.")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, we'll select the detected tabular record set @id determined above.

if len(record_sets) == 0:
    # Fallback: use records() with no argument to get the default/main table
    default_id = None
    print("Using fallback mode for main record set.")
    records = list(dataset.records())
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} records.")
    print("Columns (as field @id):")
    print(df.columns.tolist())
    df_preview = df.head()
else:
    # For each available record set, load data
    dataframes = {}
    for rsid in record_sets:
        records = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"Record set '{rsid}' loaded: {df.shape[0]} rows, {df.shape[1]} columns.")
        print('Columns:', df.columns.tolist())

    # Show head of the first record set
    chosen_record_set = record_sets[0]
    df_preview = dataframes[chosen_record_set].head()

# Display first few rows
df_preview

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This section demonstrates:
- Filtering records for a numeric field (e.g., for peptide count or molecular weight)
- Normalizing that field
- Optionally, grouping by a category field (e.g., a protein annotation)

All field access is by `@id` (Croissant field identifiers).

In [ ]:
# Try to pick common field @id names typical of protein MS tables: e.g. cr:peptide_count, cr:MW, cr:pI, cr:coverage, etc.
# (Columns in the DataFrame are exactly the field @id values as returned by Croissant parser)

# Example guesses for field @id per dataset schema documentation (replace with your field ids if documented)
import numpy as np

if len(record_sets) == 0:
    df = df
else:
    df = dataframes[chosen_record_set]

print("Available fields (by @id):", df.columns.tolist())

# Try to pick a numeric field. We'll use 'cr:MW' (molecular weight) or the first numeric column.

numeric_field_candidates = [col for col in df.columns if (
    'MW' in col or 'cr:MW' in col or 'cr:peptide_count' in col or 'coverage' in col or 'cr:coverage' in col or
    (df[col].dtype.kind in 'iufc')  # integer, unsigned, float, complex
)]

if len(numeric_field_candidates) == 0:
    # try to coerce columns to numeric to see if possible
    for col in df.columns:
        try:
            df[col+'_tmp'] = pd.to_numeric(df[col], errors='coerce')
            if df[col+'_tmp'].notna().any():
                numeric_field_candidates.append(col)
        except:
            continue
    # Clean up
    for col in df.columns:
        if col.endswith('_tmp'):
            df.drop(columns=[col], inplace=True)

if len(numeric_field_candidates) == 0:
    raise RuntimeError('No numeric fields found; unable to proceed with numeric-based EDA.')

numeric_field = numeric_field_candidates[0]
print(f"Selected numeric field for EDA: {numeric_field}")

# Coerce to numeric (if needed)
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

threshold = df[numeric_field].quantile(0.75)
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (upper quantile): {filtered_df.shape[0]} records")

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a possible category field, e.g. a protein name/descriptive field.
group_field_candidates = [col for col in df.columns if (
    'name' in col or 'cr:description' in col or 'cr:accession' in col or 'cr:protein' in col or 'description' in col
)]

if len(group_field_candidates) > 0:
    group_field = group_field_candidates[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped mean {numeric_field} by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable group field detected for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Histogram of the selected numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# If we found a grouping (e.g., for proteins/categories), show the top 10 group means
if 'grouped_df' in locals():
    plt.figure(figsize=(8,4))
    top10 = grouped_df.sort_values(by=numeric_field, ascending=False).head(10)
    sns.barplot(y=group_field, x=numeric_field, data=top10, palette='viridis')
    plt.title(f'Mean {numeric_field} by {group_field} (Top 10)')
    plt.show()

## 6. Conclusion
In this notebook, we loaded the Mass Spectrometry dataset using the `mlcroissant` library, explored its record set and fields using their `@id`s, and performed basic exploratory analysis.

**Key findings:**
1. Dataset contains protein/peptide level information with suitable numeric fields such as mass, peptide counts, or abundance data.
2. EDA reveals the distribution of these quantities and allows filtering for highly abundant/large proteins or peptides.
3. Data can be grouped by annotation fields for comparative analysis.

**Next steps** could include more focused statistical analysis, protein function annotations, or integration into downstream machine learning workflows.